## 트러블 이슈 1

- (현상) 훈련 시, 1 Epoch를 출력하기까지 10분을 초과함
    - 총 10 Epochs 예정이므로, 1시간 소요 예상

### 인간이 오래 걸린다고 느끼는 기준

|인간이 느끼는 상태|1 Epoch 기준 시간 (현재 워크로드)|주요 원인 및 심리 상태|개발 생산성 영향|
|-|-|-|-|
|즉각적 / 좋음|~30초 이내|출력이 빠르게 나오고, 기다리는 동안에도 생각을 이어갈 수 있음|최고의 몰입 상태 유지|
|안정적 (Stable)|30초 ~ 2분|커피 한 잔, Slack 확인, 간단한 메모 정도는 가능. "기다릴 만하다"는 느낌|실험 반복에 무리가 없음|
|조금 불편함|2분 ~ 5분|기다리다 보면 다른 탭을 열게 됨. "조금 느리네" 하는 수준|생산성 저하 시작|
|성가심 (Annoying)|5분 ~ 12분|기다리는 동안 집중이 완전히 깨짐. 다른 작업을 시작하게 됨|실험 주기가 길어져 동기 저하|
|고통스러움 (Painful)|12분 이상|"이걸 언제 끝나지?" 하는 생각이 지배적. 실험 자체를 꺼리게 됨|실험 횟수가 급격히 줄어듦|

- 대부분 피드백 지연이 2분을 넘기면 느리다고 느낌.
- 단순히 10 Epoch일 경우, 훈련 시간이 1시간 이상이 소요
- 시간 단축하는 여러 방법 고민 필요

### 현실적인 훈련 시간 기준 수립

1. (Step 1) 현재 워크로드 규모 요약
    - 학습 데이터: 1000장
    - batch_size=32: 약 31~32 batches per epoch
    - 모델: ResNet50
    - (소결) 전체적으로 가벼운 학습 workload (ImageNet 전체를 학습하는 수준이 아님)

2. (Step 2) 현실적인 훈련 시간 기준 수립 원칙
    - 주관적인 판단이 아니라, **하드웨어 대비 기대 성능**을 기준으로 수립 필요
    - 환경별로 명확한 threshold를 정의 필요

3. (Step 3) 실제 벤치마크 기반 추정
    - (가정) Colab T4 + PyTorch 기준으로 ResNet50 (batch 32, 224x224) 학습 시,
    - GPU에서 1 batch 처리 시간: 대략 150~400ms 수준
    - 32 batchs일 경우, 순수 연산 시간: 8~15초
    - DataLoader, tqdm, 기타 오버헤드 포함 시 실제 1 epoch 소요 시간은 25~70초 정도가 일반적임

4. (Step 4) 안정적인 영역 기준 정의
    - 해당 워크로드(ResNet50 + 1000장 + batch 32)에서 시간 기준
        |환경|1 Epoch 소요 시간|평가|판단 기준 및 행동|
        |-|-|-|-|
        |GPU (T4)|< 45초|매우 좋음 (Optimal)|이상적인 상태|
        |GPU (T4)|45초 ~ 1분 30초|안정적 (Stable)|이 구간이 목표|
        |GPU (T4)|1분 30초 ~ 3분|조금 느림|DataLoader / 설정 점검 필요|
        |GPU (T4)|3분 이상|비정상|"device 설정, precision, 병목 의심"|
        |CPU|< 10분|빠른 편|고사양 CPU|
        |CPU|10~20분|보통|일반적인 CPU 학습 속도|
        |CPU|20분 이상|느림|GPU 사용 강력 권장|
        - CPU 기준
            - 10분을 넘기면 GPU로 전환하는 것을 진지하게 고려해야 하는 수준
            - 이미 10 Epochs가 1시간 이상 소요할 것으로 예상되므로, GPU에서는 나올 수 없는 시간임.
        - GPU 기준
            - 1분 30초를 넘기면 "조금 느린" 영역에 들어섰다고 볼 수 있음.
            - 3분을 넘기면 명백히 비정상(device 설정, mixed precision 미사용, DataLoader 병목 등)
- (기준 수립) 결과적으로 **1분 내외**가 해당 워크로드에서 안정적이고 건강한 영역이므로, 이를 목표로 설정

### 해결 방안 정리

|개념 (Entity)|계층|설명|
|-|-|-|
|Infrastructure Fix|기반 환경|Colab Runtime + GPU 할당 + Device 바인딩|
|Code-level Optimization|구현 수준|device 이동, loss 함수 수정, progress bar, mixed precision"|
|Feedback Improvement|사용자 경험|tqdm, per-batch logging 등 즉각적 피드백 제공|
|Alternative Strategy|우회/대안|모델 경량화, 데이터 축소, 실험 전략 변경|
|Workflow Redesign|개발 프로세스|실험 주기를 짧게 유지하는 전체적인 접근 방식|
- 문제 해결은 단일 원인이 아니라 여러 계층의 결합으로 이루어짐
- 가장 효과적인 해결책은 **Infrastructure Fix + Code-level Optimization + Feedback Improvement 동시 적용**

1. Tier 1: 즉각 해결책 (반드시 필요)
    |순위|방안|예상 효과|난이도|비고|
    |-|-|-|-|-|
    |1|Colab Runtime을 GPU로 변경|매우 높음|낮음|가장 근본적인 해결|
    |2|코드에 device 설정 추가 (model.to(device), inputs.to(device))|매우 높음|낮음|GPU를 써도 이게 없으면 의미 없음|
    |3|Softmax 제거 (CrossEntropyLoss와 함께 사용 시 버그)|중간|낮음|학습 정확도에도 영향|
    |4|tqdm 추가 (진행률 + 실시간 loss 표시)|체감 효과 매우 높음|낮음|인간이 느끼는 "느림"을 크게 줄여줌|

2. Tier 2: 추가 성능 최적화 (Tier 1 적용 후 고려)
3. Tier 3: 대안적, 우회적 접근 (필요 시)
4. Tier 4: 개발 워크플로우 관점 해결책 (재발 방지)

- 즉각 해결책 순위 1번부터 순차적으로 진행

### 해결 방안 1 - Colab Runtime 변경

1. 사용 가능한 Colab 목록
    - Colab Pro+에서 사용 가능한 하드웨어 가속기
        - CPU
        - GPU: H100, A100, L4, T4, G4
        - TPU: v5e-1, v6e-1

2. 현재 워크로드 특성
    - 모델: ResNet50 (합성곱 연산이 지배적)
    - 데이터셋: 1000장
    - 데이터: 이미지(244 x 244)
    - 연산 특성: 행렬 곱셈 + 합성곱이 많음

3. 선택 이유
    |하드웨어|유형|CNN 학습 적합도|이 워크로드에서의 체감 속도|설정 난이도|추천도 (이번 경우)|비고|
    |-|-|-|-|-|-|-|
    |H100|GPU|최고|가장 빠름|쉬움|★★☆☆☆|과잉 + 크레딧 소모 큼|
    |A100|GPU|매우 높음|매우 빠름|쉬움|★★★☆☆|고성능이지만 과잉|
    |L4|GPU|매우 높음|매우 빠름|쉬움|★★★★☆|T4보다 신형, 성능 좋음|
    |T4 ✅|GPU|높음|빠름|쉬움|★★★★★ (강력 추천)|가장 안정적이고 무난|
    |G4|GPU|중간|보통|쉬움|★★☆☆☆|T4보다 약간 낮은 성능|
    |v5e-1 / v6e-1|TPU|중간~높음|빠름 (설정 후)|어려움|비추천|PyTorch + XLA 설정 필요|
    |CPU|CPU|매우 낮음|매우 느림|매우 쉬움|비추천|현재 상태|
    - GPU가 행렬 연산 처리 성능이 좋고, 설정 난이도가 낮음.
    - TPU는 행렬 연산 자체는 빠르지만, CNN 구조와의 호환성/최적화가 GPU만큼 좋지 않음.
    - CPU는 행렬 연산 처리 성능이 나쁨.
    - **T4 GPU** 선택
        - 과잉하지 않고, 안정적이고 무난
        - 현재 작은 워크로드이므로 T4로도 무난
        - 1 epoch 기준 30~70초 수준 예상

4. 결과 및 한계
    - (결과) 1 Epoch당 10분 &rarr; 1 Epoch당 1분 30초 감소 (약 6.67배 감소)
    - (한계) 목표는 훈련 전체가 1분 내외가 되는 게 목표이므로 추가적인 성능 개선 필요

5. 결과 근거 (CPU를 켜는 것 vs GPU를 켜는 것, Code/System Level)
    - Colab VM(가상머신)의 하드웨어 구성 단계에서 차이가 결정
        |항목|(Before) CPU Runtime|(After) GPU Runtime (T4 등)|
        |-|-|-|
        |VM 생성 시점|Google Control Plane이 CPU-only VM을 생성|Google Control Plane이 GPU가 attach된 VM을 생성|
        |하드웨어 연결|GPU가 물리적으로 연결되지 않음|NVIDIA GPU가 PCIe를 통해 VM에 연결됨|
        |OS 수준에서 보이는 장치|/dev/nvidia* 장치 파일이 존재하지 않음|/dev/nvidia0, /dev/nvidiactl, /dev/nvidia-uvm 등이 생성됨|
        |드라이버 / CUDA Runtime|NVIDIA 드라이버와 CUDA 라이브러리가 로드되지 않음|NVIDIA 드라이버 + CUDA runtime이 로드됨|
        |Python 프로세스 관점|torch.cuda.is_available() → False|torch.cuda.is_available() → True|
        |코드 실행 위치 결정 주체|OS + 하드웨어 구성|OS + 하드웨어 구성|
        - 요약
            - GPU를 켜는 것: Google이 VM에 물리적 GPU를 연결해주는 인프라 구성 작업
            - 이 단계가 끝나면 OS는 GPU를 인식하고, CUDA API 호출이 가능
        - 하지만 이 단계만으로는 PyTorch가 실제로 GPU를 사용하지 않음.

### 해결 방안 2 - 코드 내 device 설정 추가

- 코드가 GPU를 제대로 활용하지 못하거나, 다른 병목이 존재하는 상태
- 모델과 데이터를 명시적으로 GPU로 이동시킬 필요

1. 코드 내 device 설정 추가
    1. `device` 변수 설정
    2. 모델을 GPU로 이동: `.to(device)`
    3. 데이터(입력, 라벨)를 GPU로 이동: `.to(device)`

2. 결과
    - (결과) 1 Epoch당 1분 30초 &rarr; 1 Epoch당 9.78초로 감소 (약 9.2배 감소)

3. 결과 근거 (GPU를 켜는 것 vs GPU를 사용하는 것, Framework Level)
    - GPU가 OS에 보이더라도, PyTorch는 기본적으로 CPU에서 연산을 수행
    - PyTorch는 device placement라는 개념을 갖고 있음
        - 모든 torch.Tensor와 nn.Module은 어떤 device에서 관리되고 연산될지를 명시적으로 갖고 있음
        - 기본값은 cpu
        - 사용자가 명시적으로 .to("cuda") 또는 .cuda()를 호출하기 전까지는 GPU가 OS에 붙어 있어도 PyTorch는 CPU device를 사용
    - 구체적인 동작 차이
        |단계|(Before) GPU가 OS에 붙어 있는 상태 (런타임 GPU)|(After) .to(device)를 적용한 상태|
        |-|-|-|
        |Tensor 생성 위치|기본적으로 CPU 메모리 (host memory)|GPU 메모리 (device memory)|
        |연산 실행 장치|CPU에서 실행 (기본)|CUDA kernel이 GPU에서 실행|
        |데이터 이동|필요할 때마다 CPU ↔ GPU 복사 발생 (매우 비효율적)|이미 GPU에 있으므로 복사 최소화|
        |Kernel Launch|CPU에서 CUDA kernel 호출|GPU에서 직접 실행|
        |성능|GPU를 거의 사용하지 못함|GPU의 연산 능력을 제대로 사용|
        - Colab(또는 클라우드)에서 GPU를 켜는 것은 인프라 계층의 일
        - 반면 PyTorch에서 GPU를 사용하는 것은 프레임워크 실행 계층(Framework Execution Layer)의 일
        - 이 둘은 완전히 독립적인 계층이며, 아래와 같은 구조로 나뉨
            ```bash
                [Colab Control Plane]
                        ↓ (VM 생성 + GPU attach)
                [Guest OS + NVIDIA Driver + CUDA Runtime]
                        ↓ (GPU가 /dev/nvidia* 로 보임)
                [PyTorch (C++ backend + CUDA extension)]
                        ↓ (기본 device = CPU)
                [사용자 코드]
                        ↓ (.to("cuda") 호출 여부)
                실제 실행 장치 결정
            ```

### CPU 상태 비교 - Colab CPU일 때 vs Colab T4 GPU일 때

|항목|(Before) CPU Runtime|(After) GPU Runtime|차이|
|-|-|-|-|
|CPU 수|2개|12개|6배 차이|
|On-line CPU(s)|0, 1|0-11|-|
|Model name|Intel Xeon @ 2.20GHz|Intel Xeon @ 2.20GHz|동일|
|Core(s) per socket|1|6|-|
|Socket(s)|2|2|동일|
|GPU|없음|NVIDIA L4|-|

- GPU Runtime으로 변경했을 때 CPU 코어 수가 크게 증가한 것 확인 가능 (2개 -> 12개)
- 이런 차이가 발생하는 이유
    - Colab의 리소스 할당 정책
        - Google Colab은 런타임 유형에 따라 VM 스펙을 다르게 배정
        - 특히 GPU Runtime을 선택할 경우, 아래와 같은 이유로 더 많은 vCPU를 할당함
            - 딥러닝 학습에서 GPU가 연산하는 동안, CPU는 데이터를 준비해야 함(DataLoader, 이미지 전처리, augmentation 등).
            - CPU가 약하면 GPU가 기다리게 되고, 결국 GPU 활용률이 떨어짐.
            - 효과적으로 GPU를 사용하기 위해 CPU도 강력해야 할 필요.

### 트러블 이슈 1 최종 정리

1. Epoch 당 10분 초과에서 1 Epoch 당 1분 30초로 감소
   - 근거: CPU Runtime에서 GPU Runtime으로 변경했기 때문
   - 상세 설명:
        1. Runtime은 기본적으로 코드가 실행되는 가상머신 + 소프트웨어 환경 전체
        2. Runtime을 변경하면 기존 VM을 폐기하고 새로운 VM 전체를 새로 할당
        3. 변경된 새로운 VM은 GPU가 추가될 뿐 아니라, 더 좋은 성능의 CPU도 함께 할당됨
        4. 성능이 개선된 이유는 사실, GPU 때문이 아니라, GPU Runtime으로 변경하면서 더 강력한 CPU가 할당되었기 때문(아직 PyTorch가 GPU 직접 사용 전단계)

2. Epoch 당 1분 30초에서 1 Epoch 당 9.78초로 감소
    - 근거: GPU를 Framework Level에 적용했기 때문
    - 상세 설명:
        1. GPU Rumtime으로, OS에 GPU가 있더라도 PyTorch에 추가 설정을 해주지 않으면 여전히 CPU로 연산 수행
        2. PyTorch의 device placement가 기본값으로 cpu이기 때문
        3. 명시적으로 개발자가 GPU를 적용하면서, 연산 성능이 향상되었기 때문